In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import os
import argparse
import random

In [31]:
df_train = pd.read_excel('data/Training.xlsx')
df_test = pd.read_excel('data/Test-Truncated-Restated.xlsx')

In [38]:
df_train['Program'].value_counts().index[-1]

'NAZ Match/Sports Buddies'

In [40]:
df_test['Program'].value_counts()

Program
General Community                          1437
Virtual Match                               380
School-Based Program                        239
General Site                                 95
St Thomas/Four Seasons                       76
YIP 2019                                     64
Youth Leadership/YC Match                    30
BSW-Cargill                                  30
BSW-General Mills                            27
Macalester/Dayton's Bluff                    27
NAZ Zip Code                                 27
NAZ Match                                    19
St Thomas/Dayton's Bluff                     16
BSW-Carlson                                  13
Sports Buddies Cohorts 1/2 - 2023-24         12
Weaver Elementary-Maplewood PD               12
GRU Match                                    11
Columbia Heights High School Basketball      10
Grad Coach-Henry High School                 10
BSW-Comcast                                   9
Grad Coach-Edison High School   

In [41]:
df = pd.read_excel('data/Training.xlsx')
static_columns = [
    'Big Age', 
    'Big Gender', 
    'Big Race/Ethnicity',
    'Little Gender', 
    'Little Participant: Race/Ethnicity',
    'Program', 
    'Program Type',
]
cat_cols = [col for col in static_columns if df[col].dtype == 'object']
encoders = {}
for col in cat_cols:
    df[col] = df[col].fillna('unknown')
    encoder = OneHotEncoder(sparse_output=False)
    encoder.fit(df[[col]])
    least_frequent = df[col].value_counts().index[-1]
    encoders[col] = (encoder, least_frequent)


In [42]:
# Load data
def load_data(random_drop=True):
    df = pd.read_excel('data/Test-Truncated-Restated.xlsx')
    df = df.dropna(subset=['Completion Date', 'Match Support Contact Notes'])
    df['Completion Date'] = pd.to_datetime(df['Completion Date'])
    static_columns = [
    'Big Age', 
    'Big Gender', 
    'Big Race/Ethnicity',
    'Little Gender', 
    'Little Participant: Race/Ethnicity',
    'Program', 
    'Program Type',
    ]
    static_features = []
    cat_cols = [col for col in static_columns if df[col].dtype == 'object']
    num_cols = [col for col in static_columns if col not in cat_cols]
    for col in cat_cols:
        df[col] = df[col].fillna('unknown')
        encoder, least_frequent = encoders[col]
        categories = encoder.get_feature_names_out([col]).tolist()
        df.loc[~df[col].isin(categories), col] = least_frequent
        print(col)
        cat_encoded = encoder.transform(df[[col]])
        #df_encoded = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names(), index=df.index)
        df_encoded = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out([col]), index=df.index)
        df = pd.concat([df, df_encoded], axis=1).drop(columns=[col], axis=1)
        #static_features = static_features + encoder.get_feature_names_out().tolist()
        static_features += encoder.get_feature_names_out([col]).tolist()

    for col in num_cols:
        df[col] = df[col].fillna(df[col].mean())
        static_features.append(col)

    # Group and sort by Match ID and Completion Date
    grouped = df.groupby('Match ID 18Char')

    # Prepare data tuples: (sequence of notes, numerical features, target final match length)
    data = []
    for match_id, group in grouped:
        group_sorted = group.sort_values(by='Completion Date')
        notes_sequence = group_sorted['Match Support Contact Notes'].tolist()
        
        # time dependent features
        #avg_match_length = group_sorted['Match Length'].mean()
        current_match_length = (group_sorted['Completion Date'].iloc[-1] - group_sorted['Completion Date'].iloc[0]).days
        num_contacts = len(group_sorted)

        # static features
        static_values = group_sorted.iloc[0][static_features].values.astype(float)

        combined_features = [num_contacts, current_match_length] + static_values.tolist()
        #final_match_length = group_sorted['Match Length'].iloc[-1]  # Target is last match length

        data.append((notes_sequence, combined_features, match_id))

    return data, len(static_features)

In [4]:
class MatchDataset(Dataset):
     def __init__(self, data, random_drop=False):
         self.data = data
         self.random_drop = random_drop
         # Initialize SentenceTransformer model for text embedding
         self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
 
     def __len__(self):
         return len(self.data)
 
     def __getitem__(self, idx):
         notes, features, match_id = self.data[idx]
         embeddings = self.sbert.encode(notes)  # Shape: (seq_len, embed_dim)
         features = torch.tensor(features, dtype=torch.float32)
         return torch.tensor(embeddings, dtype=torch.float32), features, match_id
 

In [5]:
def collate_fn(batch):
     sequences, features, match_ids = zip(*batch)
     lengths = [seq.shape[0] for seq in sequences]
     padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True)
     features = torch.stack(features)
     return padded_sequences, torch.tensor(lengths), features, match_ids

In [6]:
class SentimentRNN(nn.Module):
     def __init__(self, embed_dim, hidden_dim, feature_dim):
         super(SentimentRNN, self).__init__()
         self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
         self.fc_1 = nn.Linear(hidden_dim + feature_dim, 128)
         self.fc_2 = nn.Linear(128, 1)
 
     def forward(self, x, lengths, features):
         packed_input = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
         packed_output, hidden = self.rnn(packed_input)
         combined = torch.cat([hidden[-1], features], dim=1)
         output = self.fc_1(combined)
         output = F.relu(output)
         output = self.fc_2(output)
         return output.squeeze()

In [50]:
def predict(model, device, loader):
    # Evaluation on test set
     model.eval()
     preds_dict = {}
     with torch.no_grad():
         for padded_seqs, lengths, features, match_ids in tqdm(loader):
             padded_seqs, lengths, features = padded_seqs.to(device), lengths.to(device), features.to(device)
             preds = model(padded_seqs, lengths, features)
             preds_list = preds.cpu().numpy()
             for pred, match_id in zip(preds_list, match_ids):
                 preds_dict[match_id] = pred
 
     return preds_dict

In [43]:
data, n_static = load_data()

Big Gender
Big Race/Ethnicity
Little Gender
Little Participant: Race/Ethnicity
Program
Program Type


In [44]:
embed_dim = 384
hidden_dim = 128
feature_dim = 2 + n_static
batch_size = 8
print(feature_dim)

102


In [45]:
dataset = MatchDataset(data)
pred_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
model = SentimentRNN(embed_dim, hidden_dim, feature_dim)
device = 'cuda' if torch.cuda.is_available() else 'cpu'




In [46]:
PATH = os.path.join(os.getcwd(), 'saved_models/rnn_model_state_dict.pth')
model.load_state_dict(torch.load(PATH, weights_only=True, map_location=torch.device('cpu')))


<All keys matched successfully>

In [51]:
pred_dict = predict(model, device, pred_loader)

100%|██████████| 38/38 [00:31<00:00,  1.19it/s]


In [52]:
print(pred_dict)

{'a1v2J0000027CXKQA2': np.float32(3.8132563), 'a1v2J0000027JFCQA2': np.float32(38.454716), 'a1v2J0000027KBoQAM': np.float32(16.331314), 'a1v2J0000027KCEQA2': np.float32(9.385356), 'a1v2J0000027KCbQAM': np.float32(13.925568), 'a1v2J0000027KWNQA2': np.float32(13.72025), 'a1v2J0000027KoQQAU': np.float32(6.142034), 'a1v2J0000027KoUQAU': np.float32(27.141714), 'a1v2J0000027KovQAE': np.float32(13.174609), 'a1v2J0000027KpFQAU': np.float32(8.652597), 'a1v2J0000027KsaQAE': np.float32(39.989117), 'a1v2J0000027LR3QAM': np.float32(45.685177), 'a1v2J0000027LRIQA2': np.float32(39.676765), 'a1v2J0000027LfeQAE': np.float32(10.842897), 'a1v2J0000027LffQAE': np.float32(25.209747), 'a1v2J0000027LflQAE': np.float32(15.352487), 'a1v2J0000027LgyQAE': np.float32(11.171937), 'a1v2J0000027LhaQAE': np.float32(9.663083), 'a1v2J0000027MG1QAM': np.float32(10.309631), 'a1v2J0000027MvnQAE': np.float32(12.315949), 'a1v2J0000027MwOQAU': np.float32(8.631075), 'a1v2J0000027Mx4QAE': np.float32(17.320158), 'a1v2J0000027NN

In [96]:
predictions_df = pd.read_csv('data/Testset_Predictions_Submit.csv', index_col='RowID')

In [97]:
predictions_df.dtypes

MatchID18Char            object
PredictedMatchLength    float64
YourTeamID              float64
dtype: object

In [98]:
predictions_df = predictions_df.astype({'MatchID18Char': str, 'PredictedMatchLength': str, 'YourTeamID': str})

In [99]:
for row in predictions_df.index:
    matchid = predictions_df.loc[row, 'MatchID18Char']
    predictions_df.loc[row, 'PredictedMatchLength'] = '{:.1f}'.format(np.round(pred_dict[matchid], 1))
    predictions_df.loc[row, 'YourTeamID'] = 'G12' 

In [100]:
predictions_df.head(300)

,MatchID18Char,PredictedMatchLength,YourTeamID
RowID,,,
1,a1v2J000003TaCwQAK,30.3,G12
2,a1v2J0000028YKuQAM,5.8,G12
3,a1v2J0000027VAVQA2,7.0,G12
4,a1v2J0000028WquQAE,52.6,G12
5,a1v2J000002ADf4QAG,13.5,G12
...,...,...,...
296,a1v2J000003A13YQAS,26.3,G12
297,a1v2J000003AMwcQAG,9.6,G12
298,a1vHt000005BXMnIAO,3.2,G12


In [101]:
predictions_df.to_csv('Testset_Predictions_Submit.csv', index=True)